# Kaggle llama.cpp API Config Runner

Notebook pendek. Semua logic ada di repo.

## Cell 1 — Clone repo + install dependency + prebuilt llama.cpp CUDA

In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/kaggle-llamacpp-api-config.git"
REPO_DIR = Path("/kaggle/working/kaggle-llamacpp-api-config")

if not REPO_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Ubah REPO_URL dulu ke repo GitHub kamu.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

from kaggle_llamacpp import ensure_llamacpp_cuda

state = ensure_llamacpp_cuda()
state


## Cell 2 — Load config + download assets

Untuk vision, pakai `config.vision.example.yaml` dan isi `model_url` + `mmproj_url` yang cocok.

In [ ]:
from pathlib import Path
from kaggle_llamacpp import load_config, print_effective_config, download_assets

CONFIG_PATH = Path("/kaggle/working/kaggle-llamacpp-api-config/configs/config.text.example.yaml")

cfg = load_config(CONFIG_PATH, profile="balanced")

# Override cepat tanpa edit file:
cfg["api"]["alias"] = "gemma-4-local"
cfg["runtime"]["ctx_size"] = 8192  # safe: 4096, balanced: 8192, long tested: 20000

print_effective_config(cfg)

assets = download_assets(cfg)
assets


## Cell 3 — Start server + health + model list + chat test

In [ ]:
from kaggle_llamacpp import start_from_config, wait_until_ready, test_models_endpoint, test_chat_completion

pid = start_from_config(cfg)

ready = wait_until_ready(cfg=cfg)
if not ready:
    raise RuntimeError("Server belum ready atau proses mati.")

models = test_models_endpoint(cfg)

text = test_chat_completion(
    cfg,
    prompt=cfg["health"]["test_prompt"],
    max_tokens=cfg["health"]["test_max_tokens"],
)
print("FINAL:", text)


## Cell 4 — Start ngrok tunnel

Pastikan secret sudah dicentang di Kaggle Secrets.

In [ ]:
from kaggle_llamacpp import start_ngrok_tunnel

public_url = start_ngrok_tunnel(
    port=cfg["api"]["port"],
    secret_name="ngrok",  # atau "NGROK_AUTHTOKEN", sesuai label secret kamu
)

print("OpenAI-compatible Base URL:")
print(public_url + "/v1")
print("Model alias:")
print(cfg["api"]["alias"])
print("API key:")
print(cfg["api"]["api_key"])


## Optional — test vision request

Hanya untuk `model.mode = vision` dan server start dengan `--mmproj`.

In [ ]:
from kaggle_llamacpp import test_vision_completion

# test_vision_completion(cfg, "/kaggle/input/your-image.jpg", "Describe this image.")
